# Week 3: Named Entity Recognition (NER)

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week03-ner.html

**How to use:** open this notebook in the course codespace (folder `notebooks/`) and run the cells from the top (Shift+Enter, or Run All). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need your key as a Codespaces secret or in the `.env` file. For Ollama examples first run `bash scripts/start-ollama.sh` in the terminal. Model-training examples are much faster on a GPU: open this notebook in Google Colab and choose Runtime → Change runtime type → T4 GPU (in Colab add API keys with the 🔑 Secrets panel).

In [ ]:
# Only needed in Google Colab (the course codespace already has these)
%pip install -q datasets seqeval

In [ ]:
# Imports that some of the lesson's code fragments assume
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

## 4. A First NER System

In [ ]:
from transformers import pipeline

checkpoint = "transformersbook/xlm-roberta-base-finetuned-panx-all"
ner = pipeline("token-classification", model=checkpoint, aggregation_strategy="simple")

sentences = [
    "Dr. Albanna teaches at the University of Petra in Amman.",
    "Angela Merkel wurde in Hamburg geboren.",
    "تقع جامعة البترا في مدينة عمّان.",
]
for s in sentences:
    print(s)
    for e in ner(s):
        print(f"   {e['entity_group']:4} {e['word']!r:32} score={e['score']:.2f}")

## A closer look at tokenisation

In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-cased")
xlmr_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")

text = "Sara teaches NER in Amman!"
bert_tokens = bert_tok(text).tokens()
xlmr_tokens = xlmr_tok(text).tokens()
print("BERT :", bert_tokens)
print("XLM-R:", xlmr_tokens)

# SentencePiece keeps the spaces, so the original text can be rebuilt exactly
print("".join(xlmr_tokens).replace("▁", " "))

## 6. The Subword Alignment Problem

In [ ]:
import pandas as pd

words = ["Dr.", "Albanna", "teaches", "at", "the", "University", "of", "Petra", "in", "Amman"]
tags  = ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O", "B-LOC"]

encoding = xlmr_tok(words, is_split_into_words=True)
tokens = xlmr_tok.convert_ids_to_tokens(encoding["input_ids"])
word_ids = encoding.word_ids()

aligned, previous = [], None
for word_id in word_ids:
    if word_id is None:                 # special token: <s> or </s>
        aligned.append("IGN")
    elif word_id == previous:           # a later subword of the same word
        aligned.append("IGN")
    else:                               # first subword of a new word
        aligned.append(tags[word_id])
    previous = word_id

print(pd.DataFrame({"token": tokens, "word_id": word_ids, "label": aligned}).to_string())

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = xlmr_tok(examples["tokens"], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        previous, label_ids = None, []
        for word_id in tokenized.word_ids(batch_index=i):
            if word_id is None or word_id == previous:
                label_ids.append(-100)
            else:
                label_ids.append(word_labels[word_id])
            previous = word_id
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

batch = {"tokens": [words], "ner_tags": [[1, 2, 0, 0, 0, 3, 4, 4, 0, 5]]}
print(tokenize_and_align_labels(batch)["labels"][0])

## 7. Evaluating an NER Model

In [ ]:
from seqeval.metrics import classification_report

y_true = [["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O", "B-LOC"],
          ["B-LOC", "O", "O", "B-PER"]]
y_pred = [["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "B-LOC", "O", "B-LOC"],
          ["B-LOC", "O", "O", "B-PER"]]

print(classification_report(y_true, y_pred))

## Evaluate the checkpoint yourself

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import classification_report, f1_score

tok = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForTokenClassification.from_pretrained(checkpoint).eval()
id2label = model.config.id2label

def predict_tags(words):
    """One predicted tag per word: the prediction for the word's first subword."""
    enc = tok(words, is_split_into_words=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        pred_ids = model(**enc).logits[0].argmax(dim=-1).tolist()
    tags, previous = [], None
    for position, word_id in enumerate(enc.word_ids()):
        if word_id is not None and word_id != previous:
            tags.append(id2label[pred_ids[position]])
        previous = word_id
    return tags

def evaluate(lang, n=200):
    test = load_dataset("unimelb-nlp/wikiann", lang, split="test").shuffle(seed=0)
    test = test.select(range(min(n, len(test))))
    names = test.features["ner_tags"].feature.names
    y_true, y_pred = [], []
    for example in test:
        pred = predict_tags(example["tokens"])
        gold = [names[i] for i in example["ner_tags"]][:len(pred)]
        y_true.append(gold)
        y_pred.append(pred)
    return y_true, y_pred

for lang in ["de", "en", "ar"]:
    y_true, y_pred = evaluate(lang)
    print(f"\n=== {lang} ===  entity-level F1 = {f1_score(y_true, y_pred):.3f}")
    print(classification_report(y_true, y_pred))

## 11. From the Lesson to Your NER Agent

In [ ]:
import re

MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
DATE_PATTERNS = [
    rf"\b\d{{1,2}} (?:{MONTHS}) \d{{4}}\b",        # 14 November 2026
    rf"\b(?:{MONTHS}) \d{{1,2}}, \d{{4}}\b",       # November 14, 2026
    rf"\b(?:{MONTHS}) \d{{4}}\b",                  # November 2026
    r"\b\d{4}-\d{2}-\d{2}\b",                      # 2026-11-14
    r"\b(?:19|20)\d{2}\b",                         # 2017
]
TECHNOLOGIES = ["XLM-RoBERTa", "BERT", "Transformer", "LangChain", "LangGraph", "PyTorch", "ChromaDB"]

def find_dates(text):
    found = []
    for pattern in DATE_PATTERNS:                   # most specific pattern first
        for m in re.finditer(pattern, text):
            overlaps = any(m.start() < e and s < m.end() for s, e, _ in found)
            if not overlaps:
                found.append((m.start(), m.end(), m.group()))
    return [{"type": "DATE", "text": t, "start": s, "end": e} for s, e, t in sorted(found)]

def find_technologies(text):
    found = []
    for term in sorted(TECHNOLOGIES, key=len, reverse=True):    # longest term first
        for m in re.finditer(rf"(?<!\w){re.escape(term)}(?!\w)", text):
            overlaps = any(m.start() < e and s < m.end() for s, e, _ in found)
            if not overlaps:
                found.append((m.start(), m.end(), m.group()))
    return [{"type": "TECH", "text": t, "start": s, "end": e} for s, e, t in sorted(found)]

article = ("The Transformer was introduced in 2017. On 14 November 2026 the team in Amman "
           "compared BERT with XLM-RoBERTa and stored the results in ChromaDB.")
for entity in find_dates(article) + find_technologies(article):
    print(entity)

## Run in your browser

**From BIO tags to entity spans, and entity-level F1**

In [ ]:
tokens = ["Dr.", "Albanna", "teaches", "at", "the", "University", "of", "Petra", "in", "Amman"]
gold   = ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O", "B-LOC"]
pred   = ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "B-LOC", "O", "B-LOC"]

def spans(tags):
    out, cur = [], None
    for i, t in enumerate(tags):
        if t.startswith("B-"):
            if cur: out.append(cur)
            cur = [t[2:], i, i]
        elif t.startswith("I-") and cur and cur[0] == t[2:]:
            cur[2] = i
        else:
            if cur: out.append(cur)
            cur = None
    if cur: out.append(cur)
    return {(typ, s, e) for typ, s, e in out}

G, P = spans(gold), spans(pred)
for typ, s, e in sorted(G | P, key=lambda x: x[1]):
    mark = "correct" if (typ, s, e) in G and (typ, s, e) in P else ("missed" if (typ, s, e) in G else "spurious")
    print(f"{typ:4} {' '.join(tokens[s:e+1]):28} {mark}")

tp = len(G & P)
precision, recall = tp / len(P), tp / len(G)
f1 = 2 * precision * recall / (precision + recall)
token_acc = sum(g == p for g, p in zip(gold, pred)) / len(gold)
print(f"\ntoken accuracy = {token_acc:.2f}, entity P={precision:.2f} R={recall:.2f} F1={f1:.2f}")

**Aligning word labels with subword tokens: where the -100 labels come from**

In [ ]:
# A toy subword tokenizer (greedy longest match) stands in for SentencePiece, so this runs in the browser.
VOCAB = {"Dr", ".", "Al", "ban", "na", "teach", "es", "at", "the", "Univers", "ity", "of", "Petra", "in", "Am", "man"}

def split_word(word):
    pieces, start = [], 0
    while start < len(word):
        end = len(word)
        while end > start and word[start:end] not in VOCAB:
            end -= 1
        if end == start:                       # no piece matches: emit one unknown character
            end = start + 1
        pieces.append(word[start:end])
        start = end
    return pieces

words = ["Dr.", "Albanna", "teaches", "at", "the", "University", "of", "Petra", "in", "Amman"]
tags  = ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O", "B-LOC"]
tag2id = {"O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4, "B-LOC": 5, "I-LOC": 6}

tokens, word_ids = ["<s>"], [None]
for i, w in enumerate(words):
    for piece in split_word(w):
        tokens.append(piece); word_ids.append(i)
tokens.append("</s>"); word_ids.append(None)

label_ids, previous = [], None
for word_id in word_ids:
    if word_id is None or word_id == previous:
        label_ids.append(-100)                 # special token, or a later subword: ignored by the loss
    else:
        label_ids.append(tag2id[tags[word_id]])
    previous = word_id

print(f"{'token':10} {'word_id':>7} {'label_id':>8}  label")
for t, w, l in zip(tokens, word_ids, label_ids):
    name = "IGN" if l == -100 else tags[w]
    print(f"{t:10} {str(w):>7} {l:>8}  {name}")

counted = sum(l != -100 for l in label_ids)
print(f"\n{len(tokens)} tokens, {len(words)} words, {counted} positions counted in the loss (one per word)")

## Test code for the course environment

**Quick NER check on your own sentence**

In [ ]:
from transformers import pipeline

ner = pipeline("token-classification",
               model="transformersbook/xlm-roberta-base-finetuned-panx-all",
               aggregation_strategy="simple")
for e in ner("Lina Haddad works for Orange Jordan in Irbid."):
    print(e["entity_group"], "|", e["word"], "|", round(float(e["score"]), 2))

**Entity-level scores with seqeval**

In [ ]:
from seqeval.metrics import classification_report

y_true = [["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "O", "B-LOC"]]
y_pred = [["B-PER", "I-PER", "O", "B-ORG", "O", "O", "B-LOC"]]
print(classification_report(y_true, y_pred))